# A2.3 · Shadow Autonomy

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Agents executing under inherited human credentials — your audit trail is already wrong.

**Control.** Detect it first, then separate the principal; revocation is impossible until you can name the actor.

**This lab.** Find agents running on inherited human credentials.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.3"))

**Shadow Autonomy** is the default failure, and it is not a bug in anyone's code. It is what happens when an agent is handed a service account and told to get on with it.

In [ ]:
from cybercommons import identity, ir
import time

good = identity.exchange(identity.mint("alice"), "patch-agent", {"repo:write"})
bad  = identity.impersonate("alice", "patch-agent", {"repo:write"})

print("delegation :", " → ".join(good.chain()))
print("impersonation:", " → ".join(bad.chain()))

Now play it forward into the incident, where the difference stops being academic.

In [ ]:
t0 = time.time()
tl = ir.Timeline()
tl.add(t0,      "alice", "alice",       "login",      "console")
tl.add(t0 + 12, "alice", "patch-agent", "write_file", "/etc/app.conf")
tl.add(t0 + 13, "alice", "patch-agent", "merge_pr",   "repo/main")
tl.add(t0 + 14, "alice", "patch-agent", "deploy",     "prod")

print("what the responder sees:")
print(tl.render())
print("\nwhat actually happened:")
print(tl.render(truth=True))

r = ir.reconstruct(tl)
print("\nattribution:", r["attribution"])
print("consequence:", r["consequence"])

### Expect

Both timelines look identical except for the actor column. `reconstruct` reports BROKEN attribution, names `patch-agent` as a hidden actor, and states that containment aimed at alice leaves the agent running.

### Your turn

Disabling alice's account is the obvious containment step and it does nothing here. Write down what the *correct* containment step is, and whether your platform can perform it today.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*